In [17]:

!pip -q install pandas convokit


In [18]:

from pathlib import Path
import pandas as pd
import json
import urllib.request
from convokit import Corpus, download

# Create project directories
BASE = Path("/content/emotion_community_project")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
RESULTS = BASE / "results"

for folder in (RAW, PROCESSED, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

# GoEmotions: read official files directly from GitHub
GOEMOTIONS_URL = (
    "https://raw.githubusercontent.com/"
    "google-research/google-research/master/"
    "goemotions/data/"
)

goemotions = {}

for split, filename in [
    ("train", "train.tsv"),
    ("validation", "dev.tsv"),
    ("test", "test.tsv"),
]:
    url = GOEMOTIONS_URL + filename
    df = pd.read_csv(url, sep="\t", header=None)

    # Official files contain comment ID, text and label IDs.

    df.columns = ["text", "labels", "comment_id"]


    df["split"] = split
    goemotions[split] = df

    print(f"GoEmotions {split}: {len(df):,} rows")

goemotions_df = pd.concat(goemotions.values(), ignore_index=True)

print("\nGoEmotions columns:", goemotions_df.columns.tolist())
print(goemotions_df.head(3))

# ConvoKit: download/load a real Reddit conversation corpus
corpus_path = download("reddit-corpus-small")
corpus = Corpus(filename=corpus_path)

convo_rows = []

for utterance in corpus.iter_utterances():
    convo_rows.append({
        "utterance_id": utterance.id,
        "text": utterance.text,
        "speaker_id": utterance.speaker.id if utterance.speaker else None,
        "reply_to": utterance.reply_to,
        "conversation_id": utterance.conversation_id,
        "timestamp": utterance.timestamp,
    })

conversations_df = pd.DataFrame(convo_rows)

print("\nConvoKit utterances:", len(conversations_df))
print("ConvoKit columns:", conversations_df.columns.tolist())
print(conversations_df.head(3))


GoEmotions train: 43,410 rows
GoEmotions validation: 5,426 rows
GoEmotions test: 5,427 rows

GoEmotions columns: ['text', 'labels', 'comment_id', 'split']
                                                text labels comment_id  split
0  My favourite food is anything I didn't have to...     27    eebbqej  train
1  Now if he does off himself, everyone will thin...     27    ed00q6i  train
2                     WHY THE FUCK IS BAYLESS ISOING      2    eezlygj  train
Dataset already exists at /root/.convokit/saved-corpora/reddit-corpus-small

ConvoKit utterances: 297132
ConvoKit columns: ['utterance_id', 'text', 'speaker_id', 'reply_to', 'conversation_id', 'timestamp']
  utterance_id                                               text  \
0       9c716m  Talk about your day. Anything goes, but subred...   
1       9c8amk  I went to visit a few days ago and Ioved it. I...   
2       9ca0yk  One time, my family and I had just returned fr...   

      speaker_id reply_to conversation_id   timest

In [19]:

print("Actual columns:", goemotions["train"].columns.tolist())
print("\nFirst 3 rows:")
display(goemotions["train"].head(3))


Actual columns: ['text', 'labels', 'comment_id', 'split']

First 3 rows:


,text,labels,comment_id,split
0,My favourite food is anything I didn't have to...,27,eebbqej,train
1,"Now if he does off himself, everyone will thin...",27,ed00q6i,train
2,WHY THE FUCK IS BAYLESS ISOING,2,eezlygj,train


In [20]:

# Standardize the column names based on the existing DataFrame
df = goemotions["train"]

if {"comment_id", "text", "labels"}.issubset(df.columns):
    df = df.rename(columns={"labels": "label_ids"})
elif len(df.columns) == 3:
    # Only use this fallback if the diagnostic confirms
    # the official order is text, labels, comment ID.
    df.columns = ["text", "label_ids", "comment_id"]

goemotions["train"] = df

print("Fixed columns:", goemotions["train"].columns.tolist())


Fixed columns: ['text', 'label_ids', 'comment_id', 'split']


In [21]:

for split, df in goemotions.items():
    if {"comment_id", "text", "labels"}.issubset(df.columns):
        goemotions[split] = df.rename(
            columns={"labels": "label_ids"}
        )

print({
    split: df.columns.tolist()
    for split, df in goemotions.items()
})


{'train': ['text', 'label_ids', 'comment_id', 'split'], 'validation': ['text', 'label_ids', 'comment_id', 'split'], 'test': ['text', 'label_ids', 'comment_id', 'split']}


In [22]:

from pathlib import Path
import json
import pandas as pd

BASE = Path("/content/emotion_community_project")
PROCESSED = BASE / "data" / "processed"
RESULTS = BASE / "results"

print("=== CLEANED FILES ===")
for path in sorted(PROCESSED.glob("*")):
    print(f"{path.name}: {path.stat().st_size:,} bytes")

report_path = RESULTS / "data_cleaning_report.json"

if report_path.exists():
    print("\n=== CLEANING REPORT ===")
    print(json.dumps(json.loads(report_path.read_text()), indent=2))
else:
    print("\nCleaning report not found. Rerun Cell 3.")

print("\n=== ROW COUNTS ===")
for name in [
    "goemotions_train.csv",
    "goemotions_validation.csv",
    "goemotions_test.csv",
    "convokit_conversations.csv",
]:
    path = PROCESSED / name
    if path.exists():
        df = pd.read_csv(path)
        print(f"{name}: {len(df):,} rows")
    else:
        print(f"{name}: NOT FOUND")


=== CLEANED FILES ===
convokit_conversations.csv: 71,595,740 bytes
goemotions_test.csv: 484,613 bytes
goemotions_test_predictions.csv: 2,476,414 bytes
goemotions_train.csv: 3,946,980 bytes
goemotions_validation.csv: 519,622 bytes

=== CLEANING REPORT ===
{
  "goemotions_train": {
    "rows_before": 43410,
    "rows_after": 43410,
    "empty_text_removed": 0,
    "duplicate_comment_ids_reported": 183,
    "rows_without_decoded_labels": 43410
  },
  "goemotions_validation": {
    "rows_before": 5426,
    "rows_after": 5426,
    "empty_text_removed": 0,
    "duplicate_comment_ids_reported": 3,
    "rows_without_decoded_labels": 5426
  },
  "goemotions_test": {
    "rows_before": 5427,
    "rows_after": 5427,
    "empty_text_removed": 0,
    "duplicate_comment_ids_reported": 6,
    "rows_without_decoded_labels": 5427
  },
  "convokit": {
    "rows_before": 297132,
    "rows_after": 292606,
    "empty_text_removed": 4526,
    "missing_speaker_ids": 0,
    "missing_reply_ids": 3763,
    "uni

In [23]:

print("GoEmotions loaded:", "goemotions" in globals())
print("ConvoKit loaded:", "conversations_df" in globals())

if "goemotions" in globals():
    for split, df in goemotions.items():
        print(split, df.shape, df.columns.tolist())


GoEmotions loaded: True
ConvoKit loaded: True
train (43410, 4) ['text', 'label_ids', 'comment_id', 'split']
validation (5426, 4) ['text', 'label_ids', 'comment_id', 'split']
test (5427, 4) ['text', 'label_ids', 'comment_id', 'split']


In [24]:

from pathlib import Path
import pandas as pd
import json
import re

BASE = Path("/content/emotion_community_project")
PROCESSED = BASE / "data" / "processed"
RESULTS = BASE / "results"

PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# Load emotion names
emotion_names = [
    line.strip()
    for line in urllib.request.urlopen(
        "https://raw.githubusercontent.com/google-research/"
        "google-research/master/goemotions/data/emotions.txt"
    ).read().decode("utf-8").splitlines()
    if line.strip()
]

def clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

report = {}

# Clean GoEmotions, preserving official splits and labels
for split, original in goemotions.items():
    df = original.copy()
    before = len(df)
    empty = df["text"].map(clean_text).eq("")
    df["text"] = df["text"].map(clean_text)

    duplicate_ids = int(df["comment_id"].duplicated().sum())
    df = df.loc[df["text"].ne("")].copy()

    def decode_labels(value):
        labels = []
        for item in str(value).split(","):
            item = item.strip()
            if item.isdigit() and int(item) < len(emotion_names):
                labels.append(emotion_names[int(item)])
        return labels

    df["emotion_labels"] = df["label_ids"].map(decode_labels)

    df.to_csv(PROCESSED / f"goemotions_{split}.csv", index=False)

    report[f"goemotions_{split}"] = {
        "rows_before": int(before),
        "rows_after": int(len(df)),
        "empty_text_removed": int(empty.sum()),
        "duplicate_comment_ids_reported": duplicate_ids,
        "rows_without_decoded_labels": int(
            df["emotion_labels"].map(len).eq(0).sum()
        ),
    }

# Clean ConvoKit data if loaded
if "conversations_df" in globals():
    conv = conversations_df.copy()
    before = len(conv)
    empty = conv["text"].map(clean_text).eq("")
    conv["text"] = conv["text"].map(clean_text)
    conv = conv.loc[conv["text"].ne("")].copy()

    conv.to_csv(PROCESSED / "convokit_conversations.csv", index=False)

    report["convokit"] = {
        "rows_before": int(before),
        "rows_after": int(len(conv)),
        "empty_text_removed": int(empty.sum()),
        "missing_speaker_ids": int(conv["speaker_id"].isna().sum()),
        "missing_reply_ids": int(conv["reply_to"].isna().sum()),
        "unique_speakers": int(conv["speaker_id"].nunique()),
        "unique_conversations": int(conv["conversation_id"].nunique()),
    }
else:
    print("ConvoKit DataFrame is not loaded; conversation cleaning skipped.")

with open(RESULTS / "data_cleaning_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("CLEANING COMPLETED")
print(json.dumps(report, indent=2))
print("\nOUTPUT FILES:")
for path in sorted(PROCESSED.glob("*.csv")):
    print(f"{path.name}: {pd.read_csv(path).shape}")
print("\nReport saved:", RESULTS / "data_cleaning_report.json")


CLEANING COMPLETED
{
  "goemotions_train": {
    "rows_before": 43410,
    "rows_after": 43410,
    "empty_text_removed": 0,
    "duplicate_comment_ids_reported": 0,
    "rows_without_decoded_labels": 0
  },
  "goemotions_validation": {
    "rows_before": 5426,
    "rows_after": 5426,
    "empty_text_removed": 0,
    "duplicate_comment_ids_reported": 0,
    "rows_without_decoded_labels": 0
  },
  "goemotions_test": {
    "rows_before": 5427,
    "rows_after": 5427,
    "empty_text_removed": 0,
    "duplicate_comment_ids_reported": 0,
    "rows_without_decoded_labels": 0
  },
  "convokit": {
    "rows_before": 297132,
    "rows_after": 292606,
    "empty_text_removed": 4526,
    "missing_speaker_ids": 0,
    "missing_reply_ids": 3763,
    "unique_speakers": 117627,
    "unique_conversations": 8286
  }
}

OUTPUT FILES:
convokit_conversations.csv: (292606, 6)
goemotions_test.csv: (5427, 5)
goemotions_test_predictions.csv: (5427, 31)
goemotions_train.csv: (43410, 5)
goemotions_validation.c

In [25]:

import json
import pandas as pd

print("=== SAVED OUTPUTS ===")

for path in sorted(PROCESSED.glob("*.csv")):
    df = pd.read_csv(path)
    print(f"{path.name}: {len(df):,} rows")

    if "text" in df.columns:
        print("  Empty texts:", df["text"].isna().sum() +
              df["text"].fillna("").str.strip().eq("").sum())

report_path = RESULTS / "data_cleaning_report.json"

print("\n=== CLEANING REPORT ===")
if report_path.exists():
    with open(report_path, encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("Report not found.")


=== SAVED OUTPUTS ===
convokit_conversations.csv: 292,606 rows
  Empty texts: 0
goemotions_test.csv: 5,427 rows
  Empty texts: 0
goemotions_test_predictions.csv: 5,427 rows
  Empty texts: 0
goemotions_train.csv: 43,410 rows
  Empty texts: 0
goemotions_validation.csv: 5,426 rows
  Empty texts: 0

=== CLEANING REPORT ===
{
  "goemotions_train": {
    "rows_before": 43410,
    "rows_after": 43410,
    "empty_text_removed": 0,
    "duplicate_comment_ids_reported": 0,
    "rows_without_decoded_labels": 0
  },
  "goemotions_validation": {
    "rows_before": 5426,
    "rows_after": 5426,
    "empty_text_removed": 0,
    "duplicate_comment_ids_reported": 0,
    "rows_without_decoded_labels": 0
  },
  "goemotions_test": {
    "rows_before": 5427,
    "rows_after": 5427,
    "empty_text_removed": 0,
    "duplicate_comment_ids_reported": 0,
    "rows_without_decoded_labels": 0
  },
  "convokit": {
    "rows_before": 297132,
    "rows_after": 292606,
    "empty_text_removed": 4526,
    "missing_sp

### Step 2: Fine-Grained Emotion Classification

In this section, we will:
1. Install the required Hugging Face libraries (`transformers`, `datasets`, `accelerate`).
2. Prepare the cleaned multi-label GoEmotions dataset.
3. Fine-tune a lightweight, fast model (`distilbert-base-uncased`) using PyTorch and the Hugging Face Trainer.
4. Predict emotion probabilities and evaluate the model using macro/micro-F1 scores.

In [26]:
!pip -q install transformers datasets accelerate scikit-learn

In [27]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

# Configure paths
BASE = Path("/content/emotion_community_project")
PROCESSED = BASE / "data" / "processed"
RESULTS = BASE / "results"

# Load cleaned datasets
train_df = pd.read_csv(PROCESSED / "goemotions_train.csv")
val_df = pd.read_csv(PROCESSED / "goemotions_validation.csv")
test_df = pd.read_csv(PROCESSED / "goemotions_test.csv")

# List of 28 GoEmotions categories (including neutral)
emotion_names = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
    'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment',
    'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism',
    'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]
num_labels = len(emotion_names)

# Convert comma-separated string labels or raw label ids back to multi-hot vectors
def get_multi_hot(label_ids_str):
    vector = np.zeros(num_labels, dtype=np.float32)
    if pd.isna(label_ids_str):
        return vector
    for idx in str(label_ids_str).split(','):
        idx = idx.strip()
        if idx.isdigit():
            val = int(idx)
            if val < num_labels:
                vector[val] = 1.0
    return vector

# Prepare labels
train_df['label_vector'] = train_df['label_ids'].apply(get_multi_hot)
val_df['label_vector'] = val_df['label_ids'].apply(get_multi_hot)
test_df['label_vector'] = test_df['label_vector'] = test_df['label_ids'].apply(get_multi_hot)

# For simple, fast execution, we will sub-sample the dataset for training demonstration
# Feel free to remove or adjust these lines for full-scale training
small_train_df = train_df.sample(2000, random_state=42).reset_index(drop=True)
small_val_df = val_df.sample(500, random_state=42).reset_index(drop=True)

# Convert to Hugging Face Dataset
def df_to_hf_dataset(df):
    return Dataset.from_dict({
        "text": df["text"].astype(str).tolist(),
        "labels": np.stack(df["label_vector"].values).tolist()
    })

hf_train = df_to_hf_dataset(small_train_df)
hf_val = df_to_hf_dataset(small_val_df)
hf_test = df_to_hf_dataset(test_df)

print(f"Demonstration Training Set Size: {len(hf_train)}")
print(f"Demonstration Validation Set Size: {len(hf_val)}")
print(f"Full Evaluation Test Set Size: {len(hf_test)}")

Demonstration Training Set Size: 2000
Demonstration Validation Set Size: 500
Full Evaluation Test Set Size: 5427


In [28]:
# Load Tokenizer and Model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = hf_train.map(tokenize_function, batched=True)
tokenized_val = hf_val.map(tokenize_function, batched=True)
tokenized_test = hf_test.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Multi-label evaluation metric helper
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Apply sigmoid to convert logits to probabilities
    probs = 1 / (1 + np.exp(-logits))
    # Select threshold of 0.5 for predictions
    preds = (probs > 0.5).astype(float)

    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
    accuracy = accuracy_score(labels, preds)

    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "accuracy": accuracy
    }

training_args = TrainingArguments(
    output_dir="./results_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# Start training
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [16]:
# Unified model training, evaluation, and saving predictions to disk
training_args = TrainingArguments(
    output_dir="./results_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("--- Starting Model Training ---")
trainer.train()

print("\n--- Evaluating and Generating Predictions ---")
predictions = trainer.predict(tokenized_test)
logits = predictions.predictions
probs = 1 / (1 + np.exp(-logits))

results_df = pd.DataFrame(test_df[["comment_id", "text", "emotion_labels"]].copy())
for i, emotion in enumerate(emotion_names):
    results_df[f"prob_{emotion}"] = probs[:, i]

# Save predicted probabilities
results_df.to_csv(PROCESSED / "goemotions_test_predictions.csv", index=False)
print("Predictions saved successfully to goemotions_test_predictions.csv!")

# Print metrics
metrics = compute_metrics((logits, predictions.label_ids))
print("\n--- Test Metrics ---")
for metric_name, score in metrics.items():
    print(f"{metric_name}: {score:.4f}")

--- Starting Model Training ---


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,Accuracy
1,0.000730,0.000426,0.000000,0.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating and Generating Predictions ---


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predictions saved successfully to goemotions_test_predictions.csv!

--- Test Metrics ---
macro_f1: 0.0000
micro_f1: 0.0000
accuracy: 1.0000
